In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)

Neo4j Python Driver Version: 6.3.1


1.เริ่มติดต่อใช้งาน Neo4j online server

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://0e6c5c98.databases.neo4j.io"
USERNAME = "0e6c5c98"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")

Password: ··········
Connected successfully


หา home database

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

for record in result.records:
    print(record.data())

{'name': '0e6c5c98', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-2a41353b6b9a-13-0140.production-orch-0695.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'constituents': []}


เก็บชื่อ database

In [ ]:
DATABASE = result.records[0]["name"]

print("Database =", DATABASE)

Database = 0e6c5c98


หลังจากนั้น Query ทุกคำสั่งใช้:

In [ ]:
result = driver.execute_query(
    """
    RETURN 'Hello Neo4j' AS message
    """,
    database_=DATABASE
)

print(result.records[0]["message"])

Hello Neo4j


ทดสอบใช้งานฐานข้อมูลจริง

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("จำนวน Node =", result.records[0]["total_nodes"])

จำนวน Node = 20


สร้าง Node

In [ ]:
result = driver.execute_query(
    """
    CREATE (u:User {
        name: 'John'
    })

    RETURN u
    """,
    database_=DATABASE
)

print(result.records[0]["u"])

ConstraintError: {neo4j_code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Node(0) already exists with label `User` and property `name` = 'John'} {gql_status: 22N79} {gql_status_description: error: data exception - property uniqueness constraint violated. Property uniqueness constraint violated: Node(0) already exists with label `User` and property `name` = 'John'.}

ตรวจสอบผลลัพธ์

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)

    RETURN
        u.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

DriverError: Driver closed

ทดลอง where

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)

    WHERE u.name = 'John'

    RETURN
        u.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

DriverError: Driver closed

6.สร้าง constrain

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_name_unique
    IF NOT EXISTS

    FOR (u:User)

    REQUIRE u.name IS UNIQUE
    """,
    database_=DATABASE
)

print("User constraint created")

DriverError: Driver closed

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT food_name_unique
    IF NOT EXISTS

    FOR (f:Food)

    REQUIRE f.name IS UNIQUE
    """,
    database_=DATABASE
)

print("Food constraint created")

DriverError: Driver closed

เพิ่ม user หลายๆคน

In [ ]:
users = [
    {"name": "John"},
    {"name": "Alice"},
    {"name": "Bob"},
    {"name": "Emma"},
    {"name": "David"},
    {"name": "Sarah"},
    {"name": "Michael"},
    {"name": "Laura"},
    {"name": "James"},
    {"name": "Emily"}
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $users AS row

    MERGE (u:User {
        name: row.name
    })

    RETURN count(u) AS total
    """,

    users=users,
    database_=DATABASE
)

print(
    "จำนวน User ที่ประมวลผล =",
    result.records[0]["total"]
)

DriverError: Driver closed

In [ ]:
foods = [
    {"name": "Basil Minced Pork"},
    {"name": "Tom Yum Goong"},
    {"name": "Crab Fried Rice"},
    {"name": "Green Curry"},
    {"name": "Pad Thai"},
    {"name": "Massaman Curry"},
    {"name": "Stir-fried Morning Glory"},
    {"name": "Papaya Salad"},
    {"name": "Khao Soi"},
    {"name": "Stir-fried Mixed Vegetables"}
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $foods AS row

    MERGE (f:Food {
        name: row.name
    })

    RETURN count(f) AS total
    """,

    foods=foods,
    database_=DATABASE
)

print(
    "จำนวน Food ที่ประมวลผล =",
    result.records[0]["total"]
)

DriverError: Driver closed

ดู user และ food ทั้งหมด

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)

    WHERE n:User OR n:Food

    RETURN
        labels(n) AS type,
        properties(n) AS data
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

DriverError: Driver closed

Step 10 — สร้าง Relationship LIKES

In [ ]:
likes = [
    {"user": "John", "food": "Basil Minced Pork"},
    {"user": "John", "food": "Green Curry"},
    {"user": "John", "food": "Papaya Salad"},
    {"user": "Alice", "food": "Basil Minced Pork"},
    {"user": "Alice", "food": "Green Curry"},
    {"user": "Alice", "food": "Tom Yum Goong"},
    {"user": "Alice", "food": "Stir-fried Morning Glory"},
    {"user": "Bob", "food": "Basil Minced Pork"},
    {"user": "Bob", "food": "Pad Thai"},
    {"user": "Bob", "food": "Khao Soi"},
    {"user": "Emma", "food": "Tom Yum Goong"},
    {"user": "Emma", "food": "Crab Fried Rice"},
    {"user": "Emma", "food": "Massaman Curry"},
    {"user": "David", "food": "Khao Soi"},
    {"user": "David", "food": "Stir-fried Morning Glory"},
    {"user": "David", "food": "Massaman Curry"},
    {"user": "Sarah", "food": "Papaya Salad"},
    {"user": "Sarah", "food": "Crab Fried Rice"},
    {"user": "Sarah", "food": "Green Curry"},
    {"user": "Michael", "food": "Stir-fried Mixed Vegetables"},
    {"user": "Michael", "food": "Pad Thai"},
    {"user": "Michael", "food": "Basil Minced Pork"},
    {"user": "Laura", "food": "Massaman Curry"},
    {"user": "Laura", "food": "Tom Yum Goong"},
    {"user": "Laura", "food": "Stir-fried Mixed Vegetables"},
    {"user": "James", "food": "Khao Soi"},
    {"user": "James", "food": "Papaya Salad"},
    {"user": "James", "food": "Stir-fried Morning Glory"},
    {"user": "Emily", "food": "Crab Fried Rice"},
    {"user": "Emily", "food": "Pad Thai"},
    {"user": "Emily", "food": "Stir-fried Mixed Vegetables"}
]

In [ ]:
driver.execute_query(
    """
    UNWIND $likes AS row

    MATCH
        (u:User {name: row.user}),
        (f:Food {name: row.food})

    MERGE
        (u)-[:LIKES]->(f)
    """,

    likes=likes,
    database_=DATABASE
)

print("สร้าง LIKES สำเร็จ")

DriverError: Driver closed

Step 11 — Query ความสัมพันธ์ ใครชอบ Pad Thai บ้าง

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[:LIKES]->
        (f:Food {name:'Pad Thai'})

    RETURN u.name AS user
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["user"])

DriverError: Driver closed

Step 13 — ดูว่าใครชอบอะไรบ้าง

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[:LIKES]->
        (f:Food)

    RETURN
        u.name AS user,
        f.name AS food

    ORDER BY user, food
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["user"], "->", record["food"])

DriverError: Driver closed

Step 14 — นำผลลัพธ์มาแสดงเป็นตาราง Pandas

In [ ]:
import pandas as pd

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[:LIKES]->
        (f:Food)

    RETURN
        u.name AS user,
        f.name AS food

    ORDER BY user
    """,
    database_=DATABASE
)

data = [
    record.data()
    for record in result.records
]

df = pd.DataFrame(data)

df

DriverError: Driver closed

Step 15 — Traversal หลายทอด

โจทย์:

คนที่ชอบอาหารเดียวกับ John แล้วยังชอบอาหารอื่นอะไรอีกบ้าง?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {name:'John'})
        -[:LIKES]->
        (sharedFood:Food)
        <-[:LIKES]-
        (similarUser:User)
        -[:LIKES]->
        (food:Food)

    WHERE
        similarUser <> me

    RETURN
        me.name AS me,
        similarUser.name AS similar_user,
        food.name AS food

    ORDER BY similar_user, food
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["me"],
        "-> คนชอบอาหารเดียวกัน:",
        record["similar_user"],
        "-> เขาชอบด้วย:",
        record["food"]
    )

Step 16 — Aggregation

โจทย์:

แต่ละคนชอบอาหารกี่เมนู?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[:LIKES]->
        (f:Food)

    RETURN
        u.name AS user,
        count(f) AS total_foods

    ORDER BY total_foods DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["user"],
        "=",
        record["total_foods"],
        "เมนู"
    )

Step 17 — อาหารเมนูไหนได้รับความนิยมมากที่สุด

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (:User)
        -[:LIKES]->
        (f:Food)

    RETURN
        f.name AS food, count(*) AS like_count

    ORDER BY like_count DESC
    """,
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

Step 18 — Mini Project: Recommendation System

โจทย์สำคัญที่สุด

แนะนำอาหารให้ John จากอาหารที่คนชอบอาหารเดียวกันเคยชอบ แต่ John ยังไม่เคยชอบ

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {name:$user_name})
        -[:LIKES]->
        (sharedFood:Food)
        <-[:LIKES]-
        (similarUser:User)
        -[:LIKES]->
        (food:Food)

    WHERE
        similarUser <> me
        AND NOT EXISTS {

            MATCH
                (me)-[:LIKES]->(food)

        }

    RETURN
        food.name AS recommendation,
        count(DISTINCT similarUser) AS similar_user_score

    ORDER BY
        similar_user_score DESC,
        recommendation
    """,

    user_name="John",
    database_=DATABASE
)

recommend_df = pd.DataFrame(
    [record.data() for record in result.records]
)

recommend_df

ทำไมบางเมนูได้คะแนนสูงกว่า?

```
เพราะ

             Alice
            /
John ─ ชอบอาหารเดียวกัน
            \
             Sarah

และ

Alice ─────── LIKES ──┐
                       ↓
              Tom Yum Goong
                       ↑
Sarah ─── LIKES ───────┘
```

ดังนั้นมีคนที่ชอบอาหารเดียวกับ John 2 คน ที่ชอบ Tom Yum Goong ด้วย

similar_user_score = 2

Step 19 — ทำเป็น Python Function

เปลี่ยน Mini Project ให้ดูเหมือนโปรแกรมจริง

In [ ]:
def recommend_foods(user_name):

    query = """
    MATCH
        (me:User {name:$user_name})
        -[:LIKES]->
        (sharedFood:Food)
        <-[:LIKES]-
        (similarUser:User)
        -[:LIKES]->
        (food:Food)

    WHERE
        similarUser <> me
        AND NOT EXISTS {

            MATCH
                (me)-[:LIKES]->(food)

        }

    RETURN
        food.name AS recommendation,
        count(DISTINCT similarUser) AS score

    ORDER BY
        score DESC,
        recommendation
    """

    result = driver.execute_query(
        query,
        user_name=user_name,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
recommend_foods("Bob")

Step 20 — Function ค้นหาผู้ใช้

In [ ]:
def find_user(user_name):

    result = driver.execute_query(
        """
        MATCH (u:User {name:$user_name})

        RETURN
            u.name AS name
        """,

        user_name=user_name,
        database_=DATABASE
    )

    if len(result.records) == 0:
        return "ไม่พบผู้ใช้"

    return result.records[0].data()

In [ ]:
find_user("Emma")

Step 21 — Function ดูอาหารที่ผู้ใช้เคยชอบ

In [ ]:
def liked_foods(user_name):

    result = driver.execute_query(
        """
        MATCH
            (u:User {name:$user_name})
            -[:LIKES]->
            (f:Food)

        RETURN
            f.name AS food

        ORDER BY food
        """,

        user_name=user_name,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
liked_foods("Bob")

Step 22 — ดู Graph ใน Neo4j Aura

หลังจากรันข้อมูลจาก Colab แล้ว ให้กลับไปที่ Neo4j Aura → Query

รัน

```
MATCH (n) RETURN n
```

เพื่อดูกราฟ User-Food ทั้งหมดแบบภาพ

In [ ]:
#Step 23 — ตรวจจำนวน Node และ Relationship
result = driver.execute_query(
    """
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())

DriverError: Driver closed

In [ ]:
#Step 24 — ปิด Connection เมื่อเรียนเสร็จ

driver.close()

print("Neo4j connection closed")

Neo4j connection closed
